In [1]:
%%capture
!pip install textacy
!pip install faiss-gpu
!pip install chromadb
!pip install nltk
!pip install sentence_transformers

In [2]:
%%capture
# Initiliase Ollama server on a thread

#Download ollama
!curl -fsSL https://ollama.com/install.sh | sh
import subprocess
process = subprocess.Popen("ollama serve", shell=True) #runs on a different thread
#Download model
!ollama pull llama3
!pip install ollama
import ollama
model_name = 'llama3'

Couldn't find '/root/.ollama/id_ed25519'. Generating new private key.
Your new public key is: 

ssh-ed25519 AAAAC3NzaC1lZDI1NTE5AAAAILznwp/xrtEPw/zMAI0c2aoJYWr4igWgIyeNbDGcLdhl



time=2025-06-26T23:04:53.594Z level=INFO source=routes.go:1235 msg="server config" env="map[CUDA_VISIBLE_DEVICES: GPU_DEVICE_ORDINAL: HIP_VISIBLE_DEVICES: HSA_OVERRIDE_GFX_VERSION: HTTPS_PROXY: HTTP_PROXY: NO_PROXY: OLLAMA_CONTEXT_LENGTH:4096 OLLAMA_DEBUG:INFO OLLAMA_FLASH_ATTENTION:false OLLAMA_GPU_OVERHEAD:0 OLLAMA_HOST:http://127.0.0.1:11434 OLLAMA_INTEL_GPU:false OLLAMA_KEEP_ALIVE:5m0s OLLAMA_KV_CACHE_TYPE: OLLAMA_LLM_LIBRARY: OLLAMA_LOAD_TIMEOUT:5m0s OLLAMA_MAX_LOADED_MODELS:0 OLLAMA_MAX_QUEUE:512 OLLAMA_MODELS:/root/.ollama/models OLLAMA_MULTIUSER_CACHE:false OLLAMA_NEW_ENGINE:false OLLAMA_NOHISTORY:false OLLAMA_NOPRUNE:false OLLAMA_NUM_PARALLEL:0 OLLAMA_ORIGINS:[http://localhost https://localhost http://localhost:* https://localhost:* http://127.0.0.1 https://127.0.0.1 http://127.0.0.1:* https://127.0.0.1:* http://0.0.0.0 https://0.0.0.0 http://0.0.0.0:* https://0.0.0.0:* app://* file://* tauri://* vscode-webview://* vscode-file://*] OLLAMA_SCHED_SPREAD:false ROCR_VISIBLE_DEVICE

[GIN] 2025/06/26 - 23:04:53 | 200 |      56.166µs |       127.0.0.1 | HEAD     "/"


time=2025-06-26T23:04:54.543Z level=INFO source=download.go:177 msg="downloading 6a0746a1ec1a in 16 291 MB part(s)"
time=2025-06-26T23:05:10.350Z level=INFO source=download.go:177 msg="downloading 4fa551d4f938 in 1 12 KB part(s)"
time=2025-06-26T23:05:11.519Z level=INFO source=download.go:177 msg="downloading 8ab4849b038c in 1 254 B part(s)"
time=2025-06-26T23:05:12.700Z level=INFO source=download.go:177 msg="downloading 577073ffcc6c in 1 110 B part(s)"
time=2025-06-26T23:05:13.845Z level=INFO source=download.go:177 msg="downloading 3f8eb4da87fa in 1 485 B part(s)"


[GIN] 2025/06/26 - 23:05:29 | 200 | 35.099433737s |       127.0.0.1 | POST     "/api/pull"


In [3]:
import re
import asyncio
import pandas as pd
import numpy as np
from pydantic import BaseModel
from ollama import AsyncClient
import textacy.datasets
import requests
import json
from sentence_transformers import SentenceTransformer
from textwrap import shorten
import nltk
from textacy.representations.vectorizers import Vectorizer
from sklearn.metrics.pairwise import cosine_similarity
import chromadb
import torch


2025-06-26 23:05:51.412118: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750979151.636976      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750979151.700713      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [4]:
legal_topics = {
    "0": "None",
    "1": "involuntary confession",
    "2": "habeas corpus",
    "3": "plea bargaining: the constitutionality of and/or the circumstances of its exercise",
    "4": "retroactivity (of newly announced or newly enacted constitutional or statutory rights)",
    "5": "search and seizure (other than as pertains to vehicles or Crime Control Act)",
    "6": "search and seizure, vehicles",
    "7": "search and seizure, Crime Control Act",
    "8": "contempt of court or congress",
    "9": "self-incrimination (other than as pertains to Miranda or immunity from prosecution)",
    "10": "Miranda warnings",
    "11": "self-incrimination, immunity from prosecution",
    "12": "right to counsel (cf. indigents appointment of counsel or inadequate representation)",
    "13": "cruel and unusual punishment, death penalty (cf. extra legal jury influence, death penalty)",
    "14": "cruel and unusual punishment, non-death penalty (cf. liability, civil rights acts)",
    "15": "line-up",
    "16": "discovery and inspection (in the context of criminal litigation only, otherwise Freedom of Information Act and related federal or state statutes or regulations)",
    "17": "double jeopardy",
    "18": "ex post facto (state)",
    "19": "extra-legal jury influences: miscellaneous",
    "20": "extra-legal jury influences: prejudicial statements or evidence",
    "21": "extra-legal jury influences: contact with jurors outside courtroom",
    "22": "extra-legal jury influences: jury instructions (not necessarily in criminal cases)",
    "23": "extra-legal jury influences: voir dire (not necessarily a criminal case)",
    "24": "extra-legal jury influences: prison garb or appearance",
    "25": "extra-legal jury influences: jurors and death penalty (cf. cruel and unusual punishment)",
    "26": "extra-legal jury influences: pretrial publicity",
    "27": "confrontation (right to confront accuser, call and cross-examine witnesses)",
    "28": "subconstitutional fair procedure: confession of error",
    "29": "subconstitutional fair procedure: conspiracy (cf. Federal Rules of Criminal Procedure: conspiracy)",
    "30": "subconstitutional fair procedure: entrapment",
    "31": "subconstitutional fair procedure: exhaustion of remedies",
    "32": "subconstitutional fair procedure: fugitive from justice",
    "33": "subconstitutional fair procedure: presentation, admissibility, or sufficiency of evidence (not necessarily a criminal case)",
    "34": "subconstitutional fair procedure: stay of execution",
    "35": "subconstitutional fair procedure: timeliness",
    "36": "subconstitutional fair procedure: miscellaneous",
    "37": "Federal Rules of Criminal Procedure",
    "38": "statutory construction of criminal laws: assault",
    "39": "statutory construction of criminal laws: bank robbery",
    "40": "statutory construction of criminal laws: conspiracy (cf. subconstitutional fair procedure: conspiracy)",
    "41": "statutory construction of criminal laws: escape from custody",
    "42": "statutory construction of criminal laws: false statements (cf. statutory construction of criminal laws: perjury)",
    "43": "statutory construction of criminal laws: financial (other than in fraud or internal revenue)",
    "44": "statutory construction of criminal laws: firearms",
    "45": "statutory construction of criminal laws: fraud",
    "46": "statutory construction of criminal laws: gambling",
    "47": "statutory construction of criminal laws: Hobbs Act; i.e., 18 USC 1951",
    "48": "statutory construction of criminal laws: immigration (cf. immigration and naturalization)",
    "49": "statutory construction of criminal laws: internal revenue (cf. Federal Taxation)",
    "50": "statutory construction of criminal laws: Mann Act and related statutes",
    "51": "statutory construction of criminal laws: narcotics includes regulation and prohibition of alcohol",
    "52": "statutory construction of criminal laws: obstruction of justice",
    "53": "statutory construction of criminal laws: perjury (other than as pertains to statutory construction of criminal laws: false statements)",
    "54": "statutory construction of criminal laws: Travel Act, 18 USC 1952",
    "55": "statutory construction of criminal laws: war crimes",
    "56": "statutory construction of criminal laws: sentencing guidelines",
    "57": "statutory construction of criminal laws: miscellaneous",
    "58": "jury trial (right to, as distinct from extra-legal jury influences)",
    "59": "speedy trial",
    "60": "miscellaneous criminal procedure (cf. due process, prisoners' rights, comity: criminal procedure)",
    "61": "voting",
    "62": "Voting Rights Act of 1965, plus amendments",
    "63": "ballot access (of candidates and political parties)",
    "64": "desegregation (other than as pertains to school desegregation, employment discrimination, and affirmative action)",
    "65": "desegregation, schools",
    "66": "employment discrimination: on basis of race, age, religion, illegitimacy, national origin, or working conditions.",
    "67": "affirmative action",
    "68": "slavery or indenture",
    "69": "sit-in demonstrations (protests against racial discrimination in places of public accommodation)",
    "70": "reapportionment: other than plans governed by the Voting Rights Act",
    "71": "debtors' rights",
    "72": "deportation (cf. immigration and naturalization)",
    "73": "employability of aliens (cf. immigration and naturalization)",
    "74": "sex discrimination (excluding sex discrimination in employment)",
    "75": "sex discrimination in employment (cf. sex discrimination)",
    "76": "Indians (other than pertains to state jurisdiction over)",
    "77": "Indians, state jurisdiction over",
    "78": "juveniles (cf. rights of illegitimates)",
    "79": "poverty law, constitutional",
    "80": "poverty law, statutory: welfare benefits, typically under some Social Security Act provision.",
    "81": "illegitimates, rights of (cf. juveniles): typically inheritance and survivor's benefits, and paternity suits",
    "82": "handicapped, rights of: under Rehabilitation, Americans with Disabilities Act, and related statutes",
    "83": "residency requirements: durational, plus discrimination against nonresidents",
    "84": "military: draftee, or person subject to induction",
    "85": "military: active duty",
    "86": "military: veteran",
    "87": "immigration and naturalization: permanent residence",
    "88": "immigration and naturalization: citizenship",
    "89": "immigration and naturalization: loss of citizenship, denaturalization",
    "90": "immigration and naturalization: access to public education",
    "91": "immigration and naturalization: welfare benefits",
    "92": "immigration and naturalization: miscellaneous",
    "93": "indigents: appointment of counsel (cf. right to counsel)",
    "94": "indigents: inadequate representation by counsel (cf. right to counsel)",
    "95": "indigents: payment of fine",
    "96": "indigents: costs or filing fees",
    "97": "indigents: U.S. Supreme Court docketing fee",
    "98": "indigents: transcript",
    "99": "indigents: assistance of psychiatrist",
    "100": "indigents: miscellaneous",
    "101": "liability, civil rights acts (cf. liability, governmental and liability, nongovernmental; cruel and unusual punishment, non-death penalty)",
    "102": "miscellaneous civil rights (cf. comity: civil rights)",
    "103": "First Amendment, miscellaneous (cf. comity: First Amendment)",
    "104": "commercial speech, excluding attorneys",
    "105": "libel, defamation: defamation of public officials and public and private persons",
    "106": "libel, privacy: true and false light invasions of privacy",
    "107": "legislative investigations: concerning internal security only",
    "108": "federal or state internal security legislation: Smith, Internal Security, and related federal statutes",
    "109": "loyalty oath or non-Communist affidavit (other than bar applicants, government employees, political party, or teacher)",
    "110": "loyalty oath: bar applicants (cf. admission to bar, state or federal or U.S. Supreme Court)",
    "111": "loyalty oath: government employees",
    "112": "loyalty oath: political party",
    "113": "loyalty oath: teachers",
    "114": "security risks: denial of benefits or dismissal of employees for reasons other than failure to meet loyalty oath requirements",
    "115": "conscientious objectors (cf. military draftee or military active duty) to military service",
    "116": "campaign spending (cf. governmental corruption):",
    "117": "protest demonstrations (other than as pertains to sit-in demonstrations): demonstrations and other forms of protest based on First Amendment guarantees",
    "118": "free exercise of religion",
    "119": "establishment of religion (other than as pertains to parochiaid:)",
    "120": "parochiaid: government aid to religious schools, or religious requirements in public schools",
    "121": "obscenity, state (cf. comity: privacy): including the regulation of sexually explicit material under the 21st Amendment",
    "122": "obscenity, federal",
    "123": "due process: miscellaneous (cf. loyalty oath), the residual code",
    "124": "due process: hearing or notice (other than as pertains to government employees or prisoners' rights)",
    "125": "due process: hearing, government employees",
    "126": "due process: prisoners' rights and defendants' rights",
    "127": "due process: impartial decision maker",
    "128": "due process: jurisdiction (jurisdiction over non-resident litigants)",
    "129": "due process: takings clause, or other non-constitutional governmental taking of property",
    "130": "privacy (cf. libel, comity: privacy)",
    "131": "abortion: including contraceptives",
    "132": "right to die",
    "133": "Freedom of Information Act and related federal or state statutes or regulations",
    "134": "attorneys' and governmental employees' or officials' fees or compensation or licenses",
    "135": "commercial speech, attorneys (cf. commercial speech)",
    "136": "admission to a state or federal bar, disbarment, and attorney discipline (cf. loyalty oath: bar applicants)",
    "137": "admission to, or disbarment from, Bar of the U.S. Supreme Court",
    "138": "arbitration (in the context of labor-management or employer-employee relations) (cf. arbitration)",
    "139": "union antitrust: legality of anticompetitive union activity",
    "140": "union or closed shop: includes agency shop litigation",
    "141": "Fair Labor Standards Act",
    "142": "Occupational Safety and Health Act",
    "143": "union-union member dispute (except as pertains to union or closed shop)",
    "144": "labor-management disputes: bargaining",
    "145": "labor-management disputes: employee discharge",
    "146": "labor-management disputes: distribution of union literature",
    "147": "labor-management disputes: representative election",
    "148": "labor-management disputes: antistrike injunction",
    "149": "labor-management disputes: jurisdictional dispute",
    "150": "labor-management disputes: right to organize",
    "151": "labor-management disputes: picketing",
    "152": "labor-management disputes: secondary activity",
    "153": "labor-management disputes: no-strike clause",
    "154": "labor-management disputes: union representatives",
    "155": "labor-management disputes: union trust funds (cf. ERISA)",
    "156": "labor-management disputes: working conditions",
    "157": "labor-management disputes: miscellaneous dispute",
    "158": "miscellaneous union",
    "159": "antitrust (except in the context of mergers and union antitrust)",
    "160": "mergers",
    "161": "bankruptcy (except in the context of priority of federal fiscal claims)",
    "162": "sufficiency of evidence: typically in the context of a jury's determination of compensation for injury or death",
    "163": "election of remedies: legal remedies available to injured persons or things",
    "164": "liability, governmental: tort or contract actions by or against government or governmental officials other than defense of criminal actions brought under a civil rights action.",
    "165": "liability, other than as in sufficiency of evidence, election of remedies, punitive damages",
    "166": "liability, punitive damages",
    "167": "Employee Retirement Income Security Act (cf. union trust funds)",
    "168": "state or local government tax",
    "169": "state and territorial land claims",
    "170": "state or local government regulation, especially of business (cf. federal pre-emption of state court jurisdiction, federal pre-emption of state legislation or regulation)",
    "171": "federal or state regulation of securities",
    "172": "natural resources - environmental protection (cf. national supremacy: natural resources, national supremacy: pollution)",
    "173": "corruption, governmental or governmental regulation of other than as in campaign spending",
    "174": "zoning: constitutionality of such ordinances, or restrictions on owners' or lessors' use of real property",
    "175": "arbitration (other than as pertains to labor-management or employeremployee relations (cf. union arbitration)",
    "176": "federal or state consumer protection: typically under the Truth in Lending; Food, Drug and Cosmetic; and Consumer Protection Credit Acts",
    "177": "patents and copyrights: patent",
    "178": "patents and copyrights: copyright",
    "179": "patents and copyrights: trademark",
    "180": "patents and copyrights: patentability of computer processes",
    "181": "federal or state regulation of transportation regulation: railroad",
    "182": "federal and some few state regulations of transportation regulation: boat",
    "183": "federal and some few state regulation of transportation regulation:truck, or motor carrier",
    "184": "federal and some few state regulation of transportation regulation: pipeline (cf. federal public utilities regulation: gas pipeline)",
    "185": "federal and some few state regulation of transportation regulation: airline",
    "186": "federal and some few state regulation of public utilities regulation: electric power",
    "187": "federal and some few state regulation of public utilities regulation: nuclear power",
    "188": "federal and some few state regulation of public utilities regulation: oil producer",
    "189": "federal and some few state regulation of public utilities regulation: gas producer",
    "190": "federal and some few state regulation of public utilities regulation: gas pipeline (cf. federal transportation regulation: pipeline)",
    "191": "federal and some few state regulation of public utilities regulation: radio and television (cf. cable television)",
    "192": "federal and some few state regulation of public utilities regulation: cable television (cf. radio and television)",
    "193": "federal and some few state regulations of public utilities regulation: telephone or telegraph company",
    "194": "miscellaneous economic regulation",
    "195": "comity: civil rights",
    "196": "comity: criminal procedure",
    "197": "comity: First Amendment",
    "198": "comity: habeas corpus",
    "199": "comity: military",
    "200": "comity: obscenity",
    "201": "comity: privacy",
    "202": "comity: miscellaneous",
    "203": "comity primarily removal cases, civil procedure (cf. comity, criminal and First Amendment); deference to foreign judicial tribunals",
    "204": "assessment of costs or damages: as part of a court order",
    "205": "Federal Rules of Civil Procedure including Supreme Court Rules, application of the Federal Rules of Evidence, Federal Rules of Appellate Procedure in civil litigation, Circuit Court Rules, and state rules and admiralty rules",
    "206": "judicial review of administrative agency's or administrative official's actions and procedures",
    "207": "mootness (cf. standing to sue: live dispute)",
    "208": "venue",
    "209": "no merits: writ improvidently granted",
    "210": "no merits: dismissed or affirmed for want of a substantial or properly presented federal question, or a nonsuit",
    "211": "no merits: dismissed or affirmed for want of jurisdiction (cf. judicial administration: Supreme Court jurisdiction or authority on appeal from federal district courts or courts of appeals)",
    "212": "no merits: adequate non-federal grounds for decision",
    "213": "no merits: remand to determine basis of state or federal court decision (cf. judicial administration: state law)",
    "214": "no merits: miscellaneous",
    "215": "standing to sue: adversary parties",
    "216": "standing to sue: direct injury",
    "217": "standing to sue: legal injury",
    "218": "standing to sue: personal injury",
    "219": "standing to sue: justiciable question",
    "220": "standing to sue: live dispute",
    "221": "standing to sue: parens patriae standing",
    "222": "standing to sue: statutory standing",
    "223": "standing to sue: private or implied cause of action",
    "224": "standing to sue: taxpayer's suit",
    "225": "standing to sue: miscellaneous",
    "226": "judicial administration: jurisdiction or authority of federal district courts or territorial courts",
    "227": "judicial administration: jurisdiction or authority of federal courts of appeals",
    "228": "judicial administration: Supreme Court jurisdiction or authority on appeal or writ of error, from federal district courts or courts of appeals (cf. 753)",
    "229": "judicial administration: Supreme Court jurisdiction or authority on appeal or writ of error, from highest state court",
    "230": "judicial administration: jurisdiction or authority of the Court of Claims",
    "231": "judicial administration: Supreme Court's original jurisdiction",
    "232": "judicial administration: review of non-final order",
    "233": "judicial administration: change in state law (cf. no merits: remand to determine basis of state court decision)",
    "234": "judicial administration: federal question (cf. no merits: dismissed for want of a substantial or properly presented federal question)",
    "235": "judicial administration: ancillary or pendent jurisdiction",
    "236": "judicial administration: extraordinary relief (e.g., mandamus, injunction)",
    "237": "judicial administration: certification (cf. objection to reason for denial of certiorari or appeal)",
    "238": "judicial administration: resolution of circuit conflict, or conflict between or among other courts",
    "239": "judicial administration: objection to reason for denial of certiorari or appeal",
    "240": "judicial administration: collateral estoppel or res judicata",
    "241": "judicial administration: interpleader",
    "242": "judicial administration: untimely filing",
    "243": "judicial administration: Act of State doctrine",
    "244": "judicial administration: miscellaneous",
    "245": "Supreme Court's certiorari, writ of error, or appeals jurisdiction",
    "246": "miscellaneous judicial power, especially diversity jurisdiction",
    "247": "federal-state ownership dispute (cf. Submerged Lands Act)",
    "248": "federal pre-emption of state court jurisdiction",
    "249": "federal pre-emption of state legislation or regulation. cf. state regulation of business. rarely involves union activity. Does not involve constitutional interpretation unless the Court says it does.",
    "250": "Submerged Lands Act (cf. federal-state ownership dispute)",
    "251": "national supremacy: commodities",
    "252": "national supremacy: intergovernmental tax immunity",
    "253": "national supremacy: marital and family relationships and property, including obligation of child support",
    "254": "national supremacy: natural resources (cf. natural resources - environmental protection)",
    "255": "national supremacy: pollution, air or water (cf. natural resources environmental protection)",
    "256": "national supremacy: public utilities (cf. federal public utilities regulation)",
    "257": "national supremacy: state tax (cf. state tax)",
    "258": "national supremacy: miscellaneous",
    "259": "miscellaneous federalism",
    "260": "boundary dispute between states",
    "261": "non-real property dispute between states",
    "262": "miscellaneous interstate relations conflict",
    "263": "incorporation of foreign territories",
    "264": "federal taxation, typically under provisions of the Internal Revenue Code",
    "265": "federal taxation of gifts, personal, business, or professional expenses",
    "266": "priority of federal fiscal claims: over those of the states or private entities",
    "267": "miscellaneous federal taxation (cf. national supremacy: state tax)",
    "268": "legislative veto",
    "269": "executive authority vis-a-vis congress or the states",
    "270": "miscellaneous",
    "271": "real property",
    "272": "personal property",
    "273": "contracts",
    "274": "evidence",
    "275": "civil procedure",
    "276": "torts",
    "277": "wills and trusts",
    "278": "commercial transactions"
} 

In [5]:
# Load Supreme Court dataset
sc = textacy.datasets.SupremeCourt()
sc.download()

# Area code classification for 15 labels
# print(sc.issue_areas)

mapper = sc.issue_area_codes

# Create a DataFrame from the dataset
# cases = [{"id": record.meta["issue"] ,"text": record.text , "actual_class": str(mapper[record.meta['issue_area']]) } for record in sc.records()]


# cases_df = pd.DataFrame(cases)

cases_df = pd.read_csv("/kaggle/input/labels-web-of-law/279_labels_data.csv")

# cases_df = cases_df[:100]
# Display the first few rows of the dataset
cases_df.head()
# cases_df = cases_df[:100]

100%|██████████| 111M/111M [00:01<00:00, 110MB/s]  


,Unnamed: 0,text,label
0,0,halliburton oil cementing v walker mrearl babc...,209
1,1,rehearing denied dec mrclaude t barnes salt la...,63
2,2,rehearing denied dec appeal district court uni...,216
3,3,mr walter j cummings jr washington dc petition...,108
4,4,mra devitt vaneck washington dc petitioner mr ...,196


In [6]:
def parse_text_to_dict(text):
    result_dict = {}
    lines = text.strip().split('\n')
    for line in lines:
        line = line.strip()
        if not line or ':' not in line:
            continue
        match = re.match(r"^(.*?):\s*(.*)$", line)
        if match:
            category = match.group(1).strip()
            values = [word.strip() for word in match.group(2).split(',')]
            result_dict[category] = values
    return result_dict

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[^a-zA-Z0-9 ]', '', text).strip().lower()

def filter_result_dict(result_dict, valid_categories):
    cleaned_valid_categories = {clean_text(str(value)): key for key, value in valid_categories.items()}
    cleaned_result_dict = {clean_text(key): values for key, values in result_dict.items()}
    final_result = {}
    for cleaned_key, values in cleaned_result_dict.items():
        if cleaned_key in cleaned_valid_categories:
            original_key = cleaned_valid_categories[cleaned_key]
            final_result[original_key] = values
    return final_result

In [7]:
# Collect results
all_results = {}

for key, topic in legal_topics.items():
    prompt = f"""
Give me a list of words and phrases (at least 10 for each) which would be present in a case text to classify it into the following category.
Do not write any explanation or warnings. This is a python style dictionary, read it and reply accordingly for the key value pair. Do not return empty results.
        {{'{topic}': []}}

Format the output in the following way:
{topic}: word1, word2, phrase1, phrase2 ...
"""
    response = ollama.chat(model=model_name, messages=[
        {'role': 'user', 'content': prompt}
    ])
    result_dict = parse_text_to_dict(response['message']['content'])
    filtered = filter_result_dict(result_dict, {key: topic for key, topic in legal_topics.items()})
    all_results.update(filtered)


time=2025-06-26T23:06:12.241Z level=INFO source=sched.go:788 msg="new model will fit in available VRAM in single GPU, loading" model=/root/.ollama/models/blobs/sha256-6a0746a1ec1aef3e7ec53868f220ff6e389f6f8ef87a01d77c96807de94ca2aa gpu=GPU-c23d3adf-c490-01aa-967f-af8c942e005d parallel=2 available=15720382464 required="6.2 GiB"
time=2025-06-26T23:06:12.450Z level=INFO source=server.go:135 msg="system memory" total="31.4 GiB" free="29.0 GiB" free_swap="0 B"
time=2025-06-26T23:06:12.451Z level=INFO source=server.go:175 msg=offload library=cuda layers.requested=-1 layers.model=33 layers.offload=33 layers.split="" memory.available="[14.6 GiB]" memory.gpu_overhead="0 B" memory.required.full="6.2 GiB" memory.required.partial="6.2 GiB" memory.required.kv="1.0 GiB" memory.required.allocations="[6.2 GiB]" memory.weights.total="4.1 GiB" memory.weights.repeating="3.7 GiB" memory.weights.nonrepeating="411.0 MiB" memory.graph.full="560.0 MiB" memory.graph.partial="677.5 MiB"
llama_model_loader: load

[GIN] 2025/06/26 - 23:06:19 | 200 |   7.75566245s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:21 | 200 |  1.473110369s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:25 | 200 |  3.982337403s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:29 | 200 |  3.950831402s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:31 | 200 |  2.233463799s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:33 | 200 |  1.867457298s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:35 | 200 |  1.876543014s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:40 | 200 |  5.294741597s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:42 | 200 |   1.80001571s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:44 | 200 |  2.079006043s |       127.0.0.1 | POST     "/api/chat"
[GIN] 2025/06/26 - 23:06:46 | 200 |  2.023770056s |       127.0.0.1 | POST     "/api/chat"

In [8]:
# Output
for category_id, terms in all_results.items():
    print(f"Category {category_id}:")
    print(terms)

Category 1:
["'forced", 'coerced', 'extracted', 'under duress', 'against will', 'compelled', 'pressured', 'intimidated', 'threatened', "blackmailed'}"]
Category 2:
["'petition'", "'writ'", "'in custody'", "'unlawful detention'", "'judicial remedy'", "'protective writ'", "'right to counsel'", "'due process violation'}"]
Category 4:
["'repeal", 'revocation', 'annulment', 'retroactive application', 'effective date', 'prospective', 'retrospective', 'applicability', 'scope', 'constitutionality', 'statutory construction', 'legislative intent', 'prior law', 'amendment', "revision'}"]
Category 5:
["'warrantless", 'probable cause', 'reasonable suspicion', 'exigent circumstances', 'knock-and-announce', 'consent search', 'search incident to arrest', 'scope of search', 'protective sweep', "inventory search'}"]
Category 6:
["['police'", "'officer'", "'vehicle stop'", "'traffic violation'", "'probable cause'", "'warrantless search'", "'reasonably suspicion'", "'motor vehicle'", "'highway patrol'", "

In [9]:
# records = list(sc.records())
# df = pd.DataFrame(records)

# # Inspect
# print(df.head())

In [10]:
import torch
from sentence_transformers import SentenceTransformer
import chromadb
import re
import gc
import pandas as pd
from tqdm import tqdm

def clean_text(text):
    if not isinstance(text, str):
        text = str(text)
    return re.sub(r'[^a-zA-Z0-9 ]', '', text).strip().lower()

def classify_case_rag(case_text, classification_dict, chroma_client, collection_name="temp_case_classification", model=None):
    """
    Dynamically classify a case using RAG by querying relevant legal topics.
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"

    # Initialize model if not passed in
    if model is None:
        model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)
        if torch.cuda.device_count() > 1:
            print(f"Using {torch.cuda.device_count()} GPUs.")
            model = torch.nn.DataParallel(model)

    # Encode helper
    def encode_text(text):
        if isinstance(text, list):
            encoded = model.module.encode(text, convert_to_tensor=True) if torch.cuda.device_count() > 1 else model.encode(text, convert_to_tensor=True)
        else:
            encoded = model.module.encode([text], convert_to_tensor=True)[0] if torch.cuda.device_count() > 1 else model.encode([text], convert_to_tensor=True)[0]
        return encoded.to(device)

    # Check and build ChromaDB collection if needed
    if collection_name not in [c.name for c in chroma_client.list_collections()]:
        collection = chroma_client.create_collection(name=collection_name)
        print("Creating new ChromaDB collection and populating...")

        for class_name, keywords in classification_dict.items():
            unique_ids = [f"{class_name}_{clean_text(k)}" for k in keywords]
            batch_embeddings = encode_text(keywords)
            collection.add(
                embeddings=[emb.cpu().tolist() for emb in batch_embeddings],
                metadatas=[{"class_name": class_name}] * len(keywords),
                ids=unique_ids
            )
    else:
        collection = chroma_client.get_collection(name=collection_name)

    # Embed case text
    case_embedding = encode_text(case_text)

    # Query top N relevant keywords
    results = collection.query(query_embeddings=[case_embedding.cpu().tolist()], n_results=3)
    retrieved_classes = [m["class_name"] for m in results["metadatas"][0]] if results["metadatas"] else []

    if not retrieved_classes:
        return "None"

    retrieved_text = "\n".join(set(retrieved_classes))

    # Prompt for LLaMA classification
    prompt = f"""
    You are an AI assistant trained in Supreme Court case classification. Given the following retrieved categories,
    classify the new case into the most appropriate class.

    Retrieved Relevant Categories:
    {retrieved_text}

    Please reply with the category number only.
    """

    response = ollama.chat(model='llama3', messages=[
        {"role": "system", "content": '''
            You are an expert in Supreme Court cases and legal classifications. Your task is to analyze the provided text of a Supreme Court case and classify it into one of the following predefined categories:

            Instructions:
            1. Carefully analyze the text and identify the primary category it belongs to.
            2. Provide only the category name as the response and nothing else.
            3. If uncertain, select the most relevant category based on the case's primary legal focus.
            4. Response should only be the class number.
        '''},
        {"role": "user", "content": prompt}
    ])

    category_response = response['message']['content'].strip()
    del case_embedding
    torch.cuda.empty_cache()
    return category_response


# -- Load dataset and classification keywords --
# records = list(sc.records())  # Your case records source
# cases_df = pd.DataFrame(records)

filtered_result = all_results

# -- Process and classify all cases --
def process_cases_rag(cases_df, filtered_result, output_file="classified_cases.csv", overwrite_collection=False):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2").to(device)
    if torch.cuda.device_count() > 1:
        model = torch.nn.DataParallel(model)

    chroma_db_path = "./chroma_db"
    chroma_client = chromadb.PersistentClient(path=chroma_db_path)

    if overwrite_collection and "temp_case_classification" in [c.name for c in chroma_client.list_collections()]:
        chroma_client.delete_collection("temp_case_classification")
        print("Deleted existing collection.")

    classified_cases = []

    for idx, row in tqdm(cases_df.iterrows(), total=len(cases_df)):
        case_text = row.get("text", "")
        if not case_text.strip():
            print(f"Skipping empty case at index {idx}")
            continue

        try:
            with torch.no_grad():
                class_name = classify_case_rag(
                    case_text, 
                    classification_dict=filtered_result, 
                    chroma_client=chroma_client, 
                    model=model
                )
            classified_cases.append({
                "id": row.get("id", f"row_{idx}"),
                "predicted_class": class_name
            })
        except Exception as e:
            print(f"Error processing case {idx}: {e}")
            continue

        torch.cuda.empty_cache()
        gc.collect()

    df_out = pd.DataFrame(classified_cases)
    df_out.to_csv(output_file, index=False)
    print(f"Saved classified results to {output_file}")


# -- Run the process --
process_cases_rag(cases_df, filtered_result, output_file="classified_cases.csv", overwrite_collection=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

100%|█████████▉| 8416/8417 [2:07:09<00:01,  1.17s/it]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[GIN] 2025/06/27 - 01:31:41 | 200 |  145.170664ms |       127.0.0.1 | POST     "/api/chat"


100%|██████████| 8417/8417 [2:07:10<00:00,  1.10it/s]

Saved classified results to classified_cases.csv


In [11]:
# Display the results
classified_df = pd.read_csv("/kaggle/working/classified_cases.csv")
classified_df.head()

cases_df["predictions"] = classified_df["predicted_class"]
cases_df.head()

,Unnamed: 0,text,label,predictions
0,0,halliburton oil cementing v walker mrearl babc...,209,5.0
1,1,rehearing denied dec mrclaude t barnes salt la...,63,2.0
2,2,rehearing denied dec appeal district court uni...,216,2.0
3,3,mr walter j cummings jr washington dc petition...,108,9.0
4,4,mra devitt vaneck washington dc petitioner mr ...,196,6.0


In [12]:
from sklearn.metrics import accuracy_score

# Ensure there are no NaNs in either column
valid_rows = cases_df.dropna(subset=["predictions", "label"])

# Compute and print accuracy
accuracy = accuracy_score(valid_rows["label"], valid_rows["predictions"])
print(f"Classification Accuracy: {accuracy:.4f}")


Classification Accuracy: 0.0056
